# Notebook 04 — Grounded Generation

Notebook 03 retrieves the right page. Now we need the *generation* step to refuse to make things up.

## Two flavors of multimodal hallucination (S3 §7.1)

1. **Fabrication** — the model describes objects/numbers that aren't in the image. "Q3 revenue was $5.2M" when the chart shows $4.2M.
2. **Omission** — the model fails to mention objects/numbers that *are* in the image. Less obvious, harder to detect.

Why this happens (§7.2):
- **Language priors dominate.** When the visual signal is weak, the LLM falls back to what's *plausible* given the question.
- **Modality fusion fades.** Visual tokens are a small slice of context; after enough text, attention to them decays.
- **Training data noise.** Web (image, caption) pairs are often loosely related — the model learns "plausible-sounding caption" as a strategy.

## What we're going to build

Three lines of defense — none of them are clever ML, all of them are engineering hygiene:

1. **Strict schema** — force the model to commit to typed fields, including a `visual_evidence` field that demands a verbatim quote of what was observed.
2. **Confidence gate** — refuse to act on low-confidence outputs downstream.
3. **Token + cost logging** — observability so you can detect drift in production.

Plus a test scaffold with mocked OpenAI responses, so the agent layer doesn't need real API calls to be tested in CI.

## 1. Setup — render one page so this notebook is self-contained

We don't need ColPali for this notebook. We'll just take page 0 of the first PDF and ask grounded questions about it.

In [ ]:
from pathlib import Path
from pdf2image import convert_from_path
import matplotlib.pyplot as plt

PDF_DIR = Path("../data/sample_pdfs")
pdfs = sorted(PDF_DIR.glob("*.pdf"))
assert pdfs, f"Drop a PDF into {PDF_DIR.resolve()}"

page = convert_from_path(str(pdfs[0]), dpi=150, first_page=1, last_page=1)[0]

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(page); ax.set_axis_off(); ax.set_title(f"{pdfs[0].name} — page 1")
plt.show()

## 2. The naive call — and how it lies

Baseline: ask GPT-4o-mini a question with no schema, no grounding instructions. Watch what happens when we ask about something that *isn't on the page*.

In [ ]:
import base64, io, os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path="../.env")
client = OpenAI()

def page_to_b64(p, q=85):
    buf = io.BytesIO(); p.save(buf, format="JPEG", quality=q)
    return base64.b64encode(buf.getvalue()).decode()

PAGE_B64 = page_to_b64(page)

def ask_naive(question: str, model: str = "gpt-4o-mini") -> str:
    r = client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{PAGE_B64}", "detail": "high"}},
                {"type": "text", "text": question},
            ],
        }],
    )
    return r.choices[0].message.content

# A question with no anchor in the page — pure language-prior bait.
# Tweak it to something *not* in your PDF. The default is intentionally generic.
BAIT = "What is the CFO's signature on this page?"
print("Q:", BAIT)
print("A:", ask_naive(BAIT))

Depending on the model and your PDF, you'll often see:
- a hedged but confident-sounding fabrication ("the signature appears to read…"), or
- the model invents a plausible name from context.

That's language-prior dominance in action. Now we fix it.

## 3. Strict schema with `visual_evidence`

The single most important defensive design choice (S3 §9.1, Step 1): a field that **forces the model to quote what it actually saw**. The model can't write a quote without grounding the claim somewhere on the page — at least not without lying twice in a row.

In [ ]:
from pydantic import BaseModel, Field, ConfigDict

class PageAnswer(BaseModel):
    """Strict-schema output for a question over a single PDF page."""
    model_config = ConfigDict(extra="ignore")  # VLMs occasionally add fields

    answer: str | None = Field(
        None,
        description="Direct answer to the question, or null if not visible on the page.",
    )
    visual_evidence: str = Field(
        ...,
        description=(
            "Verbatim quote of what was observed on the page that justifies the answer. "
            "For charts/numbers, transcribe the relevant cell/label/bar. "
            "If nothing on the page is relevant, set to 'no relevant content found'."
        ),
    )
    confidence: float = Field(
        ..., ge=0.0, le=1.0,
        description="Confidence in the answer being correct AND grounded in this page.",
    )

# Sanity-check the JSON schema you'll send to the API
import json; print(json.dumps(PageAnswer.model_json_schema(), indent=2)[:500])

## 4. The grounding system prompt

Each rule maps directly to a hallucination cause from §7.2. Self-check after writing it: which rule fights which failure mode?

In [ ]:
GROUNDING_PROMPT = """You answer questions about a single PDF page image.

RULES:
1. Only answer using information you can DIRECTLY see in the page image.
2. If the answer is not visible on the page, set `answer` to null and `confidence` below 0.3.
3. In `visual_evidence`, quote the exact text or describe the exact visual element you used.
   Generic descriptions ("a chart", "a table") are not acceptable — be specific.
4. Do NOT use prior knowledge. If the page does not say it, you do not know it.
5. For numerical answers, transcribe the source value exactly (including units, $, %).
"""

Mapping (your turn — read each rule, identify the failure mode it counters):
- Rule 1 → **fabrication** (forces direct visual grounding)
- Rule 2 → **language priors** (provides an explicit "I don't know" escape hatch)
- Rule 3 → **fabrication** (visual_evidence quote can't be generic)
- Rule 4 → **language-prior dominance**
- Rule 5 → **fabrication on numbers** (transcription forces exact reading)

## 5. The grounded call (with strict JSON schema and token logging)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger("vrag")

PRICING_PER_TOKEN = {
    "gpt-4o-mini": {"in": 0.15 / 1e6, "out": 0.60 / 1e6},
    "gpt-4o":      {"in": 2.50 / 1e6, "out": 10.0 / 1e6},
}

def ask_grounded(question: str, page_b64: str, model: str = "gpt-4o-mini") -> tuple[PageAnswer, dict]:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": GROUNDING_PROMPT},
            {"role": "user", "content": [
                {"type": "image_url",
                 "image_url": {"url": f"data:image/jpeg;base64,{page_b64}", "detail": "high"}},
                {"type": "text", "text": question},
            ]},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "page_answer",
                "strict": True,
                "schema": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["answer", "visual_evidence", "confidence"],
                    "properties": {
                        "answer": {"type": ["string", "null"]},
                        "visual_evidence": {"type": "string"},
                        "confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0},
                    },
                },
            },
        },
    )
    parsed = PageAnswer.model_validate_json(response.choices[0].message.content)
    usage = response.usage
    p = PRICING_PER_TOKEN[model]
    cost = usage.prompt_tokens * p["in"] + usage.completion_tokens * p["out"]
    metrics = {
        "model": model,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "cost_usd": round(cost, 6),
    }
    logger.info("vision_call %s", metrics)
    return parsed, metrics

## 6. See the difference

Same bait question. The naive call hallucinated. The grounded call should refuse.

In [ ]:
result, metrics = ask_grounded(BAIT, PAGE_B64)
print("Q:", BAIT)
print(result.model_dump_json(indent=2))
print("\nMetrics:", metrics)

Now try a question that **is** answerable from the page. Replace this with something specific to your PDF.

In [ ]:
REAL = "What is the title or main heading on this page?"
result, metrics = ask_grounded(REAL, PAGE_B64)
print("Q:", REAL)
print(result.model_dump_json(indent=2))
print("\nMetrics:", metrics)

## 7. The confidence gate

Grounding makes the model honest. The confidence gate makes the *system* honest — it refuses to pass low-confidence outputs to downstream consumers (a tool, a database write, a user-facing UI).

In [ ]:
CONFIDENCE_THRESHOLD = 0.5

def answer_or_refuse(question: str, page_b64: str) -> dict:
    result, metrics = ask_grounded(question, page_b64)
    if result.confidence < CONFIDENCE_THRESHOLD or result.answer is None:
        return {
            "status": "refused",
            "reason": result.visual_evidence,
            "confidence": result.confidence,
            "metrics": metrics,
        }
    return {
        "status": "ok",
        "answer": result.answer,
        "evidence": result.visual_evidence,
        "confidence": result.confidence,
        "metrics": metrics,
    }

import json
print("Bait question:")
print(json.dumps(answer_or_refuse(BAIT, PAGE_B64), indent=2, default=str))
print("\nReal question:")
print(json.dumps(answer_or_refuse(REAL, PAGE_B64), indent=2, default=str))

## 8. Test the agent layer with mocks

What separates a homework lab from a production-ready feature: **the agent layer must be testable without hitting OpenAI.** Slow, flaky, expensive otherwise.

Pattern: factor `ask_grounded` so it can be replaced with a mock that returns a `PageAnswer` directly. Run pytest in CI.

In [ ]:
from unittest.mock import patch, MagicMock

def mock_high_conf():
    return PageAnswer(
        answer="Quarterly Earnings Report — Fiscal Year 2025",
        visual_evidence="Bold heading at top of page reading 'Quarterly Earnings Report — Fiscal Year 2025'",
        confidence=0.95,
    ), {"model": "mock", "input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}

def mock_low_conf():
    return PageAnswer(
        answer=None,
        visual_evidence="no relevant content found",
        confidence=0.10,
    ), {"model": "mock", "input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}


# Test 1 — high confidence flows through
with patch("__main__.ask_grounded", side_effect=lambda *a, **kw: mock_high_conf()):
    out = answer_or_refuse("any question", "fake_b64")
    assert out["status"] == "ok"
    assert "Quarterly" in out["answer"]
    print("PASS: high-confidence answer flows through")

# Test 2 — low confidence is refused
with patch("__main__.ask_grounded", side_effect=lambda *a, **kw: mock_low_conf()):
    out = answer_or_refuse("any question", "fake_b64")
    assert out["status"] == "refused"
    print("PASS: low-confidence answer is refused — confidence gate works")

The second test is the important one. It proves the **confidence gate prevents bad data from leaving the multimodal layer** — exactly what interviewers want to see when they ask "how do you make multimodal AI safe in production?"

## What we built — production-grade Visual RAG

Combining nb03 + nb04, the full pipeline is:

```
PDF                                                    (logged)
 ├─ render → ColPali index                                ↓
 └─ render → page images cache                    [token usage]
                                                          ↓
user question → ColPali MaxSim → top-K page images
                                       ↓
                          GPT-4o w/ strict schema
                          + visual_evidence required
                                       ↓
                          confidence gate (≥ 0.5)
                                       ↓
                              {answer, evidence}
                              or {refused, reason}
```

Five techniques layered:
1. **Visual document retrieval** without OCR (ColPali, nb03)
2. **Strict-schema vision parsing** (this notebook, §3)
3. **Visual-evidence grounding** to fight fabrication (§4)
4. **Confidence gating** to prevent bad data leaking downstream (§7)
5. **Token-level cost logging** for FinOps (§5)

## Where this could extend (Phase 2+)

- **Multilingual PDFs** — ColPali's PaliGemma backbone handles Chinese/Japanese; traditional OCR doesn't. Same code, different PDF.
- **Voice queries** — Whisper transcribes the user's spoken question (multilingual + code-switching), feeds it to this same pipeline.
- **Video understanding** — frame-sample a video, ColPali-index the frames, query the same way.